In [15]:
### Preamples ###
import numpy as np
import matplotlib.pyplot as plt
import random
from numba import njit

random.seed(13)

In [16]:
def h_tilde(kappa, h):
    "h_tidle = (1 - exp(-kappa * h)) / kappa"
    return (1 - np.exp(-kappa * h)) / kappa

In [17]:
def lagm_cov_est(ret, m):
    
    N = len(ret)
    Ybar = np.mean(ret)
    
    cov = np.sum((ret[:N-m] - Ybar) * (ret[m:] - Ybar)) / (N - m)
    return cov

def estimate_kappa(ret, h , lag_M = 10):
    N = len(ret)
    
    lag1 = lagm_cov_est(ret, 1)
    
    kappa_sum = 0
    
    for m in range(2, lag_M + 1):
        lagm = lagm_cov_est(ret, m)
        
        if lagm == 0 or np.sign(lag1) != np.sign(lagm):
            continue
        
        ratio = lag1 / lagm
        if ratio <= 0 or not np.isfinite(np.log(ratio)):
            continue
        
        k_est = np.log(ratio) / ((m-1) * h)
        if not np.isfinite(k_est) or k_est <= 0:
            continue
        kappa_sum += k_est
        
    kappa = kappa_sum / (lag_M - 1)
    return kappa

In [18]:
def estimate_moments(ret):
    n = len(ret)
    Ybar = np.mean(ret)
    mean = Ybar
    var = np.var(ret, ddof = 1)
    lag1 = lagm_cov_est(ret, 1) 
    lag2 = lagm_cov_est(ret, 2)

    y2 = ret[:-1]**2
    y2d = y2 - np.mean(y2)
    yn1d = ret[1:] - np.mean(ret[1:])
    cross = np.mean(y2d * yn1d)
    
    return np.array([mean, var, lag1, lag2, cross])

In [19]:
@njit
def heston_returns(V0, N, mu, theta, kappa, sigma_V, rho, h, n_sub):
    
    dt = h / n_sub
    sqrt_dt = dt ** 0.5
    c = (1 - rho ** 2) ** 0.5 
    
    returns = np.empty(N)
  
    Vt = V0
    log_ret = 0
    
    for n in range(N):
        log_ret = 0
        for _ in range(n_sub):
            z1 = np.random.normal(0, 1)
            z2 = np.random.normal(0, 1)
            
            dB1 = sqrt_dt * z1
            dB2 = sqrt_dt * (rho * z1 + c * z2) 
            
            Vt_pos = Vt if Vt > 0 else 0 # ensure variance is non-negative for the price update
            
            log_ret += (mu - 0.5 * Vt) * dt + Vt_pos ** 0.5 * dB1 # log return increment
            
            Vt = Vt_pos + kappa * (theta - Vt_pos)* dt + sigma_V * Vt_pos ** 0.5 * dB2  # variance process
            
        returns[n] = log_ret
        
    return returns 

In [20]:
def estimate_heston_from_returns(ret, h):
    N = len(ret)
    mean_e, var_e, lag1_e, lag2_e, cross_e = estimate_moments(ret)
    
    kappa = estimate_kappa(ret, h)
    print( f"  κ̂    = {kappa:.4f}")
    
    ht = h_tilde(kappa, h)
    
    ek = np.exp(-kappa * h)
    
    dh = h * ek - ht
    
    theta = var_e / h  - (2* (h-ht) / (h * kappa * ht ** 2)) * lag1_e 
    print(f"  θ̂    = {theta:.4f}")
    
    mu = mean_e / h + theta / 2
    print(f"  μ̂    = {mu:.4f}")
    
    num   = (4*kappa*mean_e
             + (8*dh/(theta*ht**3))*lag1_e
             - 2*kappa*cross_e/lag1_e)
    den   = (theta*ht**2/(2*lag1_e) - dh/(kappa*ht))

    sigma_sq = num / den
    sigma_V = sigma_sq**0.5
    print(f"  σ̂_V  = {sigma_V:.6f}")

    rho = np.clip(sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
                  -0.999, 0.999)
    print(f"  ρ̂    = {rho:.6f}")

    return {'mu': mu, 'theta': theta, 'kappa': kappa,
            'sigma_V': sigma_V, 'rho': rho}

In [21]:
def estimate_moments(ret):
    n = len(ret)
    Ybar = np.mean(ret)
    mean = Ybar
    var = np.var(ret, ddof = 1)
    lag1 = lagm_cov_est(ret, 1) 
    lag2 = lagm_cov_est(ret, 2)

    y2 = ret[:-1]**2
    y2d = y2 - np.mean(y2)
    yn1d = ret[1:] - np.mean(ret[1:])
    cross = np.mean(y2d * yn1d)
    
    return np.array([mean, var, lag1, lag2, cross])

In [22]:
def estimate_from_moments(samp, h):
    mean_e, var_e, lag1_e, lag2_e, cross_e = samp
    
    kappa = -np.log(lag2_e / lag1_e) / h
    
    ht      = (1 - np.exp(-kappa*h)) / kappa
    ek      = np.exp(-kappa*h)
    dh      = h*ek - ht
    theta   = var_e/h - (2*(h-ht)/(h*kappa*ht**2))*lag1_e
    mu      = mean_e/h + theta/2
    num     = (4*kappa*mean_e + (8*dh/(theta*ht**3))*lag1_e
               - 2*kappa*cross_e/lag1_e)
    den     = theta*ht**2/(2*lag1_e) - dh/(kappa*ht)
    sigma_V = (num/den)**0.5
    rho     = np.clip(sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
                      -0.999, 0.999)
    return {'kappa': kappa, 'theta': theta, 'sigma_V': sigma_V,
            'mu': mu, 'rho': rho}

In [23]:
def jacobian(samp, est, h):
    k = len(samp)
    Jg  = np.zeros((5,5))
    
    for j in range(k):
        eps = max(abs(samp[j]) * 1e-4, 1e-8)
        
        samp_up     = samp.copy()
        samp_dn     = samp.copy()
        samp_up[j]  += eps
        samp_dn[j]  -= eps
        
        try:
            est_up = estimate_from_moments(samp_up, h)
            est_dn = estimate_from_moments(samp_dn, h)
            for i, key in enumerate(['kappa', 'theta', 'sigma_V', 'mu', 'rho']):
                Jg[i, j] = (est_up[key] - est_dn[key]) / (2 * eps)
        except Exception as ex:
            print(f"  Jacobian column {j} failed: {ex}")
        
    return Jg

In [24]:
def newey_west(z, lag_max):
    
    N, k = z.shape
    Sigma = (z.T @ z) / N # lag-0 term
    
    for lag in range(1, lag_max + 1):
        w = 1 - lag / (lag_max + 1)
        Gamma = (z[lag:].T @ z[:-lag]) / N
        Sigma += w * (Gamma + Gamma.T)
        
    return Sigma

def moment_conditions(ret, params):
    mu, theta, kappa, sigma_V, rho = params
    N = len(ret)
    Ybar = np.mean(ret)
    Y2bar = np.mean(ret ** 2)
    
    # Moments per obs
    z = np.zeros((N, 5))
    z[:, 0] = ret - Ybar                                    # mean
    z[:, 1] = (ret - Ybar) ** 2 - np.var(ret, ddof = 1)     # var
    cov1 = lagm_cov_est(ret, 1)
    z[:-1, 2] = (ret[:-1] - Ybar) * (ret[1:] - Ybar) - cov1     # lag1

    cov2 = lagm_cov_est(ret, 2)
    z[:-2, 3] = (ret[:-2] - Ybar) * (ret[2:] - Ybar) - cov2     # lag2

    y2    = ret[:-1]**2
    y2bar = np.mean(y2)
    cross = np.mean((y2 - y2bar) * (ret[1:] - np.mean(ret[1:])))
    z[:-1, 4] = (y2 - y2bar) * (ret[1:] - Ybar) - cross         # cross
    return z

In [25]:
from scipy.stats import norm
def confidence_intervals(ret, est, h, alpha = 0.05):
    N = len(ret)
    samp = estimate_moments(ret)
    
    # Estimate Sigma via Newey-West
    params = [est['mu'], est['theta'], est['kappa'], est['sigma_V'], est['rho'] ]
    z      = moment_conditions(ret, params)
    lag_nw = int(4 * (N/100)**(2/9)) # bandwidth
    Sigma  = newey_west(z, lag_nw)
    
    # Numerical computation of the Jacobian
    Jg = jacobian(samp, est, h)
    
    # Asymptotic covariance of parameter estimates
    V = Jg @ Sigma @ Jg.T
    se = np.sqrt(np.diag(V) / N) # standard errors
    
    # Critical value
    z_crit  = norm.ppf(1 - alpha / 2)
    
    keys   = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    TRUE_v = [TRUE['kappa'], TRUE['theta'], TRUE['sigma_V'], TRUE['mu'], TRUE['rho']]

    print(f"\n{'Parameter':<10s}  {'Estimate':>10s}  {'Std Err':>10s}  "
          f"{'95% CI Lower':>14s}  {'95% CI Upper':>14s}  {'True':>8s}  {'Covered':>8s}")
    print("-" * 82)

    results = {}
    for i, key in enumerate(keys):
        e   = est[key]
        s   = se[i]
        lo  = e - z_crit * s
        hi  = e + z_crit * s
        t   = TRUE_v[i]
        cov = '✓' if lo <= t <= hi else '✗'
        print(f"  {key:<10s}  {e:10.4f}  {s:10.4f}  {lo:14.4f}  {hi:14.4f}  "
              f"{t:8.4f}  {cov:>8s}")
        results[key] = (e, lo, hi, s)

    return results

In [ ]:
# Change of parameter

def experiment_coverage(TRUE, V0, h, n_sub, n_rep=100, N=int(400_000)):
    """
    For each replication, compute the 95% CI and check if the true
    value is covered. Reports empirical coverage probability.
    Should be close to 95% if asymptotic theory is correct.
    """
    keys     = ['kappa', 'theta', 'sigma_V', 'mu', 'rho']
    covered  = {key: 0 for key in keys}
    ci_width = {key: [] for key in keys}
    total    = 0

    print(f"Running {n_rep} replications for coverage experiment …")

    for r in range(n_rep):
        ret = heston_returns(V0, N,
                             TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                             TRUE['sigma_V'], TRUE['rho'], h, n_sub)
        try:
            est = estimate_heston_from_returns(ret, h)
            ci  = confidence_intervals(ret, est, h, alpha=0.05)
            total += 1
            for key in keys:
                e, lo, hi, se = ci[key]
                if lo <= TRUE[key] <= hi:
                    covered[key] += 1
                ci_width[key].append(hi - lo)
        except Exception as ex:
            print(f"  Rep {r+1} failed: {ex}")

    print(f"\n{'Parameter':<12s}  {'True':>8s}  {'Coverage':>10s}  "
          f"{'Target':>8s}  {'Mean CI Width':>14s}")
    print("-" * 58)
    for key in keys:
        cov   = covered[key] / total * 100
        width = np.nanmean(ci_width[key])
        flag  = '✓' if abs(cov - 95) < 5 else '✗'
        print(f"  {key:<10s}  {TRUE[key]:8.4f}  {cov:9.1f}%  "
              f"{'95.0%':>8s}  {width:14.4f}  {flag}")
    return covered, ci_width

In [ ]:
# ── Run ────────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)

experiment_coverage(TRUE, V0, h, n_sub, n_rep=200, N=int(400_000))

Running 200 replications for coverage experiment …
  κ̂    = 0.1121
  θ̂    = 0.2513
  μ̂    = 0.1241
  σ̂_V  = 0.109420
  ρ̂    = -0.682767

Parameter     Estimate     Std Err    95% CI Lower    95% CI Upper      True   Covered
----------------------------------------------------------------------------------
  kappa           0.1121      0.0568          0.0007          0.2235    0.1000         ✓
  theta           0.2513      0.0012          0.2489          0.2537    0.2500         ✓
  sigma_V         0.1094      0.0318          0.0472          0.1717    0.1000         ✓
  mu              0.1241      0.0010          0.1222          0.1259    0.1250         ✓
  rho            -0.6828      0.1542         -0.9851         -0.3805   -0.7000         ✓
  κ̂    = 0.1033
  θ̂    = 0.2499
  μ̂    = 0.1244
  σ̂_V  = 0.103955
  ρ̂    = -0.739733

Parameter     Estimate     Std Err    95% CI Lower    95% CI Upper      True   Covered
-----------------------------------------------------------------

C:\Users\karis\AppData\Local\Temp\ipykernel_52552\513026693.py:14: RuntimeWarning: invalid value encountered in scalar power
  sigma_V = (num/den)**0.5



Parameter     Estimate     Std Err    95% CI Lower    95% CI Upper      True   Covered
----------------------------------------------------------------------------------
  kappa           0.0831      0.0571         -0.0289          0.1951    0.1000         ✓
  theta           0.2497      0.0012          0.2474          0.2520    0.2500         ✓
  sigma_V         0.0857         nan             nan             nan    0.1000         ✗
  mu              0.1266      0.0009          0.1248          0.1285    0.1250         ✓
  rho            -0.8017         nan             nan             nan   -0.7000         ✗
  κ̂    = 0.0878
  θ̂    = 0.2504
  μ̂    = 0.1256
  σ̂_V  = 0.088465
  ρ̂    = -0.754822

Parameter     Estimate     Std Err    95% CI Lower    95% CI Upper      True   Covered
----------------------------------------------------------------------------------
  kappa           0.0878      0.0610         -0.0317          0.2073    0.1000         ✓
  theta           0.2504      0.00

({'kappa': 200, 'theta': 186, 'sigma_V': 193, 'mu': 190, 'rho': 171},
 {'kappa': [np.float64(0.22278494869619658),
   np.float64(0.21565442443018507),
   np.float64(0.22878603960086363),
   np.float64(0.22368349923195147),
   np.float64(0.23565806100159864),
   np.float64(0.23233504025589546),
   np.float64(0.22873952863595753),
   np.float64(0.23634398992903405),
   np.float64(0.2298492985582448),
   np.float64(0.25497527503699874),
   np.float64(0.23449306381549057),
   np.float64(0.21887413517920454),
   np.float64(0.22400211864490213),
   np.float64(0.23904139742656583),
   np.float64(0.2195894936127989),
   np.float64(0.24160765421133068),
   np.float64(0.23839437295910557),
   np.float64(0.21743063208767455),
   np.float64(0.21906329772165),
   np.float64(0.21656998136052896),
   np.float64(0.2458177798548463),
   np.float64(0.21635685017921372),
   np.float64(0.21710805065248118),
   np.float64(0.2267694835874603),
   np.float64(0.22571683823104077),
   np.float64(0.219723974572

In [ ]:
def bootstrap_ci(ret, est, h, n_boot=200, n_sub=20, alpha=0.05):
    """
    Parametric bootstrap CI — the practical alternative to
    replication-based intervals when true parameters are unknown.
    
    Procedure:
      1. Treat est as if it were the true parameter vector
      2. Simulate n_boot paths from the fitted model
      3. Re-estimate from each bootstrap path
      4. Use the distribution of bootstrap estimates as the CI
    
    """
    from scipy.stats import norm

    keys     = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']
    boot_est = {k: [] for k in keys}
    N        = len(ret)
    V0       = est['theta']   # start in stationarity
    failed   = 0

    print(f"Running {n_boot} bootstrap replications …")

    for b in range(n_boot):
        # Simulate from FITTED model (not true model)
        ret_boot = heston_returns(V0, N,
                                  est['mu'], est['theta'], est['kappa'],
                                  est['sigma_V'], est['rho'],
                                  h, n_sub)
        try:
            est_boot = estimate_heston_from_returns(ret_boot, h)
            for k in keys:
                boot_est[k].append(est_boot[k])
        except Exception:
            failed += 1

        if (b+1) % 50 == 0:
            print(f"  {b+1}/{n_boot}  failed: {failed}")

    print(f"\nCompleted: {n_boot-failed}/{n_boot}\n")

    z_crit  = norm.ppf(1 - alpha/2)
    results = {}

    print(f"{'Parameter':<12s}  {'Estimate':>10s}  {'Boot Std':>10s}  "
          f"{'95% CI Lower':>14s}  {'95% CI Upper':>14s}  {'Width':>8s}")
    print("-" * 72)

    for key in keys:
        vals = np.array(boot_est[key])
        e    = est[key]
        s    = vals.std()              # bootstrap std
        lo   = e - z_crit * s         # normal bootstrap CI
        hi   = e + z_crit * s

        # Alternatively — percentile bootstrap (more robust)
        lo_pct = np.percentile(vals, 100*alpha/2)
        hi_pct = np.percentile(vals, 100*(1-alpha/2))

        print(f"  {key:<10s}  {e:10.4f}  {s:10.4f}  "
              f"{lo:14.4f}  {hi:14.4f}  {hi-lo:8.4f}")
        results[key] = (e, lo, hi, s, lo_pct, hi_pct)

    return results

In [30]:
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)
N     = int(400_000)

# Step 1 — simulate one path (your "observed" data)
ret = heston_returns(V0, N,
                     TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                     TRUE['sigma_V'], TRUE['rho'], h, n_sub)

# Step 2 — estimate parameters from that single path
est = estimate_heston_from_returns(ret, h)

print("Point estimates:")
for k, v in est.items():
    print(f"  {k:<10s} = {v:.4f}  (true = {TRUE[k]:.4f})")

# Step 3 — bootstrap CI using the estimated parameters
boot = bootstrap_ci(ret, est, h, n_boot=200, n_sub=n_sub, alpha=0.05)

# Step 4 — compare all three CI types side by side
print(f"\n{'Parameter':<12s}  {'True':>8s}  {'Estimate':>10s}  "
      f"{'Delta CI':>20s}  {'Bootstrap CI':>20s}  {'Boot Width':>10s}")
print("-" * 86)

# Get delta method CI for comparison
ci_delta = confidence_intervals(ret, est, h, alpha=0.05)

for key in ['mu', 'theta', 'kappa', 'sigma_V', 'rho']:
    true     = TRUE[key]
    e        = est[key]
    d_lo, d_hi = ci_delta[key][1], ci_delta[key][2]
    b_lo, b_hi = boot[key][1],     boot[key][2]
    print(f"  {key:<10s}  {true:8.4f}  {e:10.4f}  "
          f"({d_lo:7.4f}, {d_hi:7.4f})  "
          f"({b_lo:7.4f}, {b_hi:7.4f})  "
          f"{b_hi-b_lo:10.4f}")

  κ̂    = 0.0855
  θ̂    = 0.2500
  μ̂    = 0.1248
  σ̂_V  = 0.092314
  ρ̂    = -0.735131
Point estimates:
  mu         = 0.1248  (true = 0.1250)
  theta      = 0.2500  (true = 0.2500)
  kappa      = 0.0855  (true = 0.1000)
  sigma_V    = 0.0923  (true = 0.1000)
  rho        = -0.7351  (true = -0.7000)
Running 200 bootstrap replications …
  κ̂    = 0.0757
  θ̂    = 0.2499
  μ̂    = 0.1231
  σ̂_V  = 0.088662
  ρ̂    = -0.791072
  κ̂    = 0.0897
  θ̂    = 0.2504
  μ̂    = 0.1240
  σ̂_V  = 0.091643
  ρ̂    = -0.773802
  κ̂    = 0.0767
  θ̂    = 0.2490
  μ̂    = 0.1253
  σ̂_V  = 0.086664
  ρ̂    = -0.774102
  κ̂    = 0.0877
  θ̂    = 0.2490
  μ̂    = 0.1243
  σ̂_V  = 0.094206
  ρ̂    = -0.722089
  κ̂    = 0.0943
  θ̂    = 0.2492
  μ̂    = 0.1245
  σ̂_V  = 0.094407
  ρ̂    = -0.739515
  κ̂    = 0.0668
  θ̂    = 0.2482
  μ̂    = 0.1266
  σ̂_V  = 0.077004
  ρ̂    = -0.816131
  κ̂    = 0.0865
  θ̂    = 0.2491
  μ̂    = 0.1256
  σ̂_V  = 0.096462
  ρ̂    = -0.712045
  κ̂    = 0.0980
  θ̂    = 0.